In [11]:
!pip install wfdb

In [12]:
!git clone https://github.com/muha-0/beat-synchronous-tokenizer

fatal: destination path 'beat-synchronous-tokenizer' already exists and is not an empty directory.


In [13]:
%cd beat-synchronous-tokenizer
!ls

/content/beat-synchronous-tokenizer/beat-synchronous-tokenizer
beat-synchronous-tokenizer	ecg_ssl    train.py
checkpoints_fixed_ssl_rawpatch	notebooks


In [14]:
!mkdir -p physionet.org/files/ptb-xl/1.0.3
!wget -O physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv
!wget -O physionet.org/files/ptb-xl/1.0.3/scp_statements.csv https://physionet.org/files/ptb-xl/1.0.3/scp_statements.csv

--2026-04-01 01:37:42--  https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv
Resolving physionet.org (physionet.org)... 18.18.42.54
Connecting to physionet.org (physionet.org)|18.18.42.54|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6594879 (6.3M) [text/plain]
Saving to: ‘physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv’

physionet.org/files 100%[===================>]   6.29M   243KB/s    in 26s     

2026-04-01 01:38:09 (244 KB/s) - ‘physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv’ saved [6594879/6594879]

--2026-04-01 01:38:09--  https://physionet.org/files/ptb-xl/1.0.3/scp_statements.csv
Resolving physionet.org (physionet.org)... 18.18.42.54
Connecting to physionet.org (physionet.org)|18.18.42.54|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9720 (9.5K) [text/plain]
Saving to: ‘physionet.org/files/ptb-xl/1.0.3/scp_statements.csv’

physionet.org/files 100%[===================>]   9.49K  --.-KB/s    in 0s      

2

In [15]:
import sys
sys.path.append('.')
from pathlib import Path
import ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import wfdb
from sklearn.metrics import roc_auc_score, f1_score, classification_report

from ecg_ssl.model import ECGMaskedSSL
from ecg_ssl import config

In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [17]:
pretrained_model = ECGMaskedSSL(
    in_channels=config.IN_CHANNELS,
    seq_len=config.SEQ_LEN,
    d_model=config.D_MODEL,
    patch_size=config.PATCH_SIZE,
    num_heads=config.NUM_HEADS,
    num_layers=config.NUM_LAYERS,
    mlp_ratio=config.MLP_RATIO,
    dropout=config.DROPOUT,
).to(device)

checkpoint_path = Path("checkpoints_fixed_ssl_rawpatch/best.pt")
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
pretrained_model.load_state_dict(checkpoint["model_state_dict"])
pretrained_model.eval()
print("loaded checkpoint from: ", checkpoint_path)


loaded checkpoint from:  checkpoints_fixed_ssl_rawpatch/best.pt


In [18]:
class PTBXLClassifier(nn.Module):
  def __init__(self, pretrained_model, feature_dim = 256, num_classes=1):
    super().__init__()
    self.pretrained_model = pretrained_model
    self.classifier = nn.Linear(feature_dim, num_classes)

  def forward(self, x):
    out = self.pretrained_model(x)
    pooled = out["pooled"]
    logits = self.classifier(pooled)
    return logits




In [19]:
model = PTBXLClassifier(pretrained_model).to(device)
print(model)

PTBXLClassifier(
  (pretrained_model): ECGMaskedSSL(
    (tokenizer): FixedCNNTokenizer(
      (proj): Conv1d(12, 256, kernel_size=(50,), stride=(50,))
    )
    (posenc): LearnablePositionalEncoding()
    (encoder): TransformerEncoder(
      (encoder): TransformerEncoder(
        (layers): ModuleList(
          (0-3): 4 x TransformerEncoderLayer(
            (self_attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
            )
            (linear1): Linear(in_features=256, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
            (linear2): Linear(in_features=1024, out_features=256, bias=True)
            (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
            (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
            (dropout1): Dropout(p=0.1, inplace=False)
            (dropout2): Dropout(p=0.1, inplace=False)
          )
        )


In [20]:
ptbxl_path = Path("physionet.org/files/ptb-xl/1.0.3")

label_df = pd.read_csv(ptbxl_path / "ptbxl_database.csv")
scp_df = pd.read_csv(ptbxl_path / "scp_statements.csv", index_col=0)

print("label_df shape:", label_df.shape)
print("scp_df shape:", scp_df.shape)
display(label_df.head())
display(scp_df.head())

label_df shape: (21799, 28)
scp_df shape: (71, 12)


,ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
0,1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
1,2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
2,3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,...,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr
3,4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,...,True,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr
4,5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,...,True,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr


,description,diagnostic,form,rhythm,diagnostic_class,diagnostic_subclass,Statement Category,SCP-ECG Statement Description,AHA code,aECG REFID,CDISC Code,DICOM Code
NDT,non-diagnostic T abnormalities,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,non-diagnostic T abnormalities,NaN,NaN,NaN,NaN
NST_,non-specific ST changes,1.0,1.0,NaN,STTC,NST_,Basic roots for coding ST-T changes and abnorm...,non-specific ST changes,145.0,MDC_ECG_RHY_STHILOST,NaN,NaN
DIG,digitalis-effect,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,suggests digitalis-effect,205.0,NaN,NaN,NaN
LNGQT,long QT-interval,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,long QT-interval,148.0,NaN,NaN,NaN
NORM,normal ECG,1.0,NaN,NaN,NORM,NORM,Normal/abnormal,normal ECG,1.0,NaN,NaN,F-000B7


In [21]:
label_df["scp_codes"] = label_df["scp_codes"].apply(ast.literal_eval)
label_df[["scp_codes"]].head()

,scp_codes
0,"{'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}"
1,"{'NORM': 80.0, 'SBRAD': 0.0}"
2,"{'NORM': 100.0, 'SR': 0.0}"
3,"{'NORM': 100.0, 'SR': 0.0}"
4,"{'NORM': 100.0, 'SR': 0.0}"


In [22]:
def is_normal_ecg(code_dict):
    return int("NORM" in code_dict)

label_df["target_norm"] = label_df["scp_codes"].apply(is_normal_ecg)

print(label_df["target_norm"].value_counts())

target_norm
0    12285
1     9514
Name: count, dtype: int64


In [23]:
ptbxl_download_dir = "ptbxl_download"

sample_records = label_df["filename_hr"].head(500).tolist()

wfdb.dl_database(
    "ptb-xl",
    dl_dir=ptbxl_download_dir,
    records=sample_records,
)

print("Done")

Generating record list for: records500/00000/00001_hr
Generating record list for: records500/00000/00002_hr
Generating record list for: records500/00000/00003_hr
Generating record list for: records500/00000/00004_hr
Generating record list for: records500/00000/00005_hr
Generating record list for: records500/00000/00006_hr
Generating record list for: records500/00000/00007_hr
Generating record list for: records500/00000/00008_hr
Generating record list for: records500/00000/00009_hr
Generating record list for: records500/00000/00010_hr
Generating record list for: records500/00000/00011_hr
Generating record list for: records500/00000/00012_hr
Generating record list for: records500/00000/00013_hr
Generating record list for: records500/00000/00014_hr
Generating record list for: records500/00000/00015_hr
Generating record list for: records500/00000/00016_hr
Generating record list for: records500/00000/00017_hr
Generating record list for: records500/00000/00018_hr
Generating record list for: 

In [24]:
!find ptbxl_download -name "*_hr.hea" | wc -l
!find ptbxl_download -name "*_hr.dat" | wc -l

500
500


In [25]:
ptbxl_path = Path("ptbxl_download")

In [26]:
def get_ecg_path_hr(row):
    return ptbxl_path / row["filename_hr"]

label_df["ecg_path"] = label_df.apply(get_ecg_path_hr, axis=1)

def files_exist(row):
    base = Path(row["ecg_path"])
    return Path(str(base) + ".hea").exists() and Path(str(base) + ".dat").exists()

label_df["file_exists"] = label_df.apply(files_exist, axis=1)
print(label_df["file_exists"].value_counts())

file_exists
False    21299
True       500
Name: count, dtype: int64


In [27]:
label_df_small = label_df[label_df["file_exists"]].copy()

print("Remaining rows:", len(label_df_small))
print(label_df_small["target_norm"].value_counts())

Remaining rows: 500
target_norm
1    297
0    203
Name: count, dtype: int64


In [28]:
class PTBXLDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        record_path = str(row["ecg_path"])

        record = wfdb.rdrecord(record_path)
        x = record.p_signal.astype(np.float32).T   # (12, 5000)

        mean = x.mean(axis=1, keepdims=True)
        std = x.std(axis=1, keepdims=True)
        x = (x - mean) / np.clip(std, 1e-4, None)
        x = np.clip(x, -5, 5)

        x = torch.from_numpy(x).float()
        y = torch.tensor(row["target_norm"], dtype=torch.float32)

        return x, y

In [29]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    label_df_small,
    test_size=0.30,
    random_state=42,
    stratify=label_df_small["target_norm"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=None,
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

print("\nTrain label counts:")
print(train_df["target_norm"].value_counts())

print("\nVal label counts:")
print(val_df["target_norm"].value_counts())

print("\nTest label counts:")
print(test_df["target_norm"].value_counts())

Train: 350
Val: 75
Test: 75


In [30]:
train_dataset = PTBXLDataset(train_df)
val_dataset = PTBXLDataset(val_df)
test_dataset = PTBXLDataset(test_df)

batch_size = 8

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print("Dataloaders ready")

Dataloaders ready


In [37]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5, weight_decay=.05)

In [38]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device).unsqueeze(1)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [39]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device).unsqueeze(1)

            logits = model(x)
            loss = criterion(logits, y)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            total_loss += loss.item()
            all_probs.extend(probs.cpu().numpy().flatten())
            all_preds.extend(preds.cpu().numpy().flatten())
            all_labels.extend(y.cpu().numpy().flatten())

    auc = roc_auc_score(all_labels, all_probs)
    f1 = f1_score(all_labels, all_preds)

    return total_loss / len(loader), auc, f1



In [40]:
num_epochs = 3

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_auc, val_f1 = evaluate(model, val_loader, criterion, device)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val AUC:    {val_auc:.4f}")
    print(f"Val F1:     {val_f1:.4f}")
    print("-" * 40)

Epoch 1/3
Train Loss: 0.2816
Val Loss:   0.4803
Val AUC:    0.8644
Val F1:     0.7907
----------------------------------------
Epoch 2/3
Train Loss: 0.2773
Val Loss:   0.4582
Val AUC:    0.8615
Val F1:     0.8571
----------------------------------------
Epoch 3/3
Train Loss: 0.2479
Val Loss:   0.4933
Val AUC:    0.8452
Val F1:     0.8602
----------------------------------------


In [41]:
test_loss, test_auc, test_f1 = evaluate(model, test_loader, criterion, device)

print("Test Loss:", round(test_loss, 4))
print("Test AUC:", round(test_auc, 4))
print("Test F1:", round(test_f1, 4))

Test Loss: 0.4852
Test AUC: 0.8695
Test F1: 0.8132


References to remember for report, we can delete this at the end I just dont want to forget

Goldberger, A., Amaral, L., Glass, L., Hausdorff, J., Ivanov, P. C., Mark, R., ... & Stanley, H. E. (2000). PhysioBank, PhysioToolkit, and PhysioNet: Components of a new research resource for complex physiologic signals. Circulation [Online]. 101 (23), pp. e215–e220. RRID:SCR_007345.


Wagner, P., Strodthoff, N., Bousseljot, R., Samek, W., & Schaeffter, T. (2022). PTB-XL, a large publicly available electrocardiography dataset (version 1.0.3). PhysioNet. RRID:SCR_007345. https://doi.org/10.13026/kfzx-aw45


Wagner, P., Strodthoff, N., Bousseljot, R.-D., Kreiseler, D., Lunze, F.I., Samek, W., Schaeffter, T. (2020), PTB-XL: A Large Publicly Available ECG Dataset. Scientific Data. https://doi.org/10.1038/s41597-020-0495-6

